# 05 — Comparação de estratégias e recomendação

Este notebook consolida as evidências dos notebooks 02, 03 e 04 (não redescobre nada, só compara o que já foi
medido) para decidir qual das duas estratégias de target no Medical Abstracts TC Corpus seguir na comparação de
modelos (notebook 06).

In [1]:
import json

import pandas as pd

with open("../data/experiments/results/eda_medical_abstracts_summary.json") as f:
    eda_ma = json.load(f)
with open("../data/experiments/results/baseline_results.json") as f:
    baseline = json.load(f)

print("Resumos carregados dos notebooks 03 e 04 (nenhum número novo é inventado aqui).")

Resumos carregados dos notebooks 03 e 04 (nenhum número novo é inventado aqui).


## Tabela comparativa

In [2]:
s2 = baseline["strategy02_urgency"]
s3 = baseline["strategy03_clean"]
s3_official = baseline["strategy03_official"]

rows = {
    "Target": ["urgency (normal/atenção/urgente)", "condition_label (especialidade clínica)"],
    "Nº de classes": [3, 5],
    "Origem do split": ["próprio (notebook 02, estratificado, sem leakage)",
                         "oficial da fonte, com overlap ambíguo removido do teste (notebook 03/04)"],
    "Linhas de treino": [s2["n_train"], s3["n_train"]],
    "Linhas de teste": [s2["n_test"], s3["n_test"]],
    "Accuracy (split honesto)": [s2["accuracy"], s3["accuracy"]],
    "Macro F1 (split honesto)": [s2["macro_f1"], s3["macro_f1"]],
    "Accuracy (split oficial, sem tratamento)": ["n/a — split já é próprio, sem overlap por construção",
                                                  s3_official["accuracy"]],
    "Tempo de treino baseline (s)": [s2["train_time_s"], s3["train_time_s"]],
    "Latência de inferência (ms, média)": [s2["inference_latency_ms_mean"], s3["inference_latency_ms_mean"]],
    "Tamanho do modelo (KB)": [s2["model_size_kb"], s3["model_size_kb"]],
    "Bate literalmente com o target pedido pelo Tech Challenge (urgência)?": ["Sim", "Não (aceito como alternativa)"],
}

comparison = pd.DataFrame(rows, index=["Estratégia 02 (urgência)", "Estratégia 03 (condição médica)"]).T
comparison

,Estratégia 02 (urgência),Estratégia 03 (condição médica)
Target,urgency (normal/atenção/urgente),condition_label (especialidade clínica)
Nº de classes,3,5
Origem do split,"próprio (notebook 02, estratificado, sem leakage)","oficial da fonte, com overlap ambíguo removido..."
Linhas de treino,8981,11550
Linhas de teste,2246,1878
Accuracy (split honesto),0.7306,0.7295
Macro F1 (split honesto),0.7197,0.7118
"Accuracy (split oficial, sem tratamento)","n/a — split já é próprio, sem overlap por cons...",0.4744
Tempo de treino baseline (s),3.512,22.716
"Latência de inferência (ms, média)",14.5758,14.3497


## Qual estratégia parece mais adequada, e por quê?

### Leitura dos números

- **Qualidade de classificação:** praticamente empatada. Estratégia 02 tem accuracy 73,1% / macro F1 0,720;
  Estratégia 03 (split limpo) tem accuracy 73,0% / macro F1 0,712 — diferença de menos de 1 ponto percentual,
  dentro da margem que se espera de ruído entre splits diferentes. **Nenhuma das duas estratégias é
  claramente superior em performance de modelo.**
- **Custo computacional:** aqui a diferença é real. Estratégia 02 treina ~6,5× mais rápido (3,5s vs 22,7s) e
  gera um modelo quase 2× menor (64 MB vs 126 MB) — resultado direto de ter menos classes (3 vs 5) e menos
  linhas de treino (o split próprio reparticiona o corpus inteiro 80/20, enquanto o split oficial já vem com
  mais linhas do lado do treino).
- **Adequação ao enunciado:** a Estratégia 02 entrega o target de urgência pedido literalmente pelo Tech
  Challenge, sem precisar de nenhuma exceção. A Estratégia 03 usa `condition_label` nativo (especialidade
  clínica), que só é válido como target se a equipe/professor aceitar essa substituição explicitamente.
- **Risco de dado:** a Estratégia 03 depende de uma limpeza (remoção de pares ambíguos do teste) que precisa ser
  reaplicada com cuidado sempre que o dataset for retreinado; a Estratégia 02 já nasce sem esse problema porque
  o split é construído do zero com verificação de disjunção.

### Recomendação

Como as métricas de qualidade estão estatisticamente empatadas, a decisão entre as duas estratégias é
**principalmente um critério de produto/enunciado, não um critério de modelo** — ver a pergunta ao usuário no
fechamento deste notebook antes de fixar a estratégia usada nos notebooks 06 e 07.

In [3]:
summary = {
    "strategy02": {
        "target": "urgency (3 classes)",
        "accuracy": s2["accuracy"],
        "macro_f1": s2["macro_f1"],
        "train_time_s": s2["train_time_s"],
        "model_size_kb": s2["model_size_kb"],
        "matches_challenge_target_literally": True,
    },
    "strategy03": {
        "target": "condition_label (5 classes)",
        "accuracy": s3["accuracy"],
        "macro_f1": s3["macro_f1"],
        "train_time_s": s3["train_time_s"],
        "model_size_kb": s3["model_size_kb"],
        "matches_challenge_target_literally": False,
        "requires_explicit_exception": "aceitar condição médica no lugar de urgência",
    },
    "metrics_verdict": "empate estatístico (diferença < 1 p.p. em accuracy e macro F1)",
    "decisive_factor": "produto/enunciado (qual target faz sentido), não performance de modelo",
    "recommended_strategy": None,
}

with open("../data/experiments/results/dataset_recommendation.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

summary

{'strategy02': {'target': 'urgency (3 classes)',
  'accuracy': 0.7306,
  'macro_f1': 0.7197,
  'train_time_s': 3.512,
  'model_size_kb': 65952.7,
  'matches_challenge_target_literally': True},
 'strategy03': {'target': 'condition_label (5 classes)',
  'accuracy': 0.7295,
  'macro_f1': 0.7118,
  'train_time_s': 22.716,
  'model_size_kb': 128658.9,
  'matches_challenge_target_literally': False,
  'requires_explicit_exception': 'aceitar condição médica no lugar de urgência'},
 'metrics_verdict': 'empate estatístico (diferença < 1 p.p. em accuracy e macro F1)',
 'decisive_factor': 'produto/enunciado (qual target faz sentido), não performance de modelo',
 'recommended_strategy': None}